---
## Task 4 — Shots: Winning Teams vs. Losing Teams

### Analytic question formulation
Do **winning teams** register significantly more **shots per match** than **losing teams**?
(Shot volume is a common proxy for attacking output, so we expect winners to out-shoot losers.)

**H0:** mean shots(winning team) = mean shots(losing team)
**H1:** mean shots(winning team) > mean shots(losing team)

### Data wrangling
Load the raw match-level CSV, drop drawn matches (there is no "winner" to compare), and derive
two paired columns: the shot count for the match's winning team and the shot count for its
losing team.

In [8]:
from google.colab import drive
import pandas as pd
import numpy as np
import scipy.stats as st

drive.mount('/content/drive')

file_path = '/content/drive/MyDrive/Colab/world_cup_match_data.csv'

df = pd.read_csv(file_path)
print("CSV imported successfully!")
print(df.head())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
CSV imported successfully!
    timestamp              date_GMT    status  attendance home_team_name  \
0  1781204400  Jun 11 2026 - 7:00pm  complete       80824         Mexico   
1  1781229600  Jun 12 2026 - 2:00am  complete       44985    South Korea   
2  1781290800  Jun 12 2026 - 7:00pm  complete       43002         Canada   
3  1781312400  Jun 13 2026 - 1:00am  complete       70492          USMNT   
4  1781377200  Jun 13 2026 - 7:00pm  complete       67966          Qatar   

           away_team_name  referee  Game Week  Pre-Match PPG (Home)  \
0            South Africa      NaN        1.0                   0.0   
1          Czech Republic      NaN        1.0                   0.0   
2  Bosnia and Herzegovina      NaN        1.0                   0.0   
3                Paraguay      NaN        1.0                   0.0   
4             Switzerland      N

In [9]:
# Remove draws — a decisive result is required to define a winner/loser
df_shots = df[
    df['home_team_goal_count'] != df['away_team_goal_count']
].copy()

# Winning and losing team shots
win_team_shots = df_shots.apply(
    lambda x: x['home_team_shots']
    if x['home_team_goal_count'] > x['away_team_goal_count']
    else x['away_team_shots'],
    axis=1
)

lose_team_shots = df_shots.apply(
    lambda x: x['away_team_shots']
    if x['home_team_goal_count'] > x['away_team_goal_count']
    else x['home_team_shots'],
    axis=1
)

print("Population size (decisive matches):", len(df_shots))

Population size (decisive matches): 80


### Data preparation and sampling
**Population:** one winner-shots value and one loser-shots value per decisive match.
**Sample:** an independent **simple random sample of n = 30** drawn separately from the
winning-team shot counts and the losing-team shot counts, for a two-sample comparison.

In [10]:
n = 30
win_sample  = win_team_shots.sample(n=n, random_state=7).reset_index(drop=True)
lose_sample = lose_team_shots.sample(n=n, random_state=7).reset_index(drop=True)

print("Winners sample n:", len(win_sample), " Losers sample n:", len(lose_sample))

Winners sample n: 30  Losers sample n: 30


### Descriptive statistics

In [11]:
for name, s in [("Winning teams", win_sample), ("Losing teams", lose_sample)]:
    print(f"\n{name}:")
    print(s.describe())


Winning teams:
count    30.000000
mean     13.933333
std       5.570447
min       6.000000
25%       9.000000
50%      13.500000
75%      19.000000
max      23.000000
dtype: float64

Losing teams:
count    30.000000
mean     10.100000
std       5.683309
min       2.000000
25%       6.250000
50%       9.000000
75%      12.500000
max      32.000000
dtype: float64


### Inferential statistics — Confidence intervals (95%, each group)

In [12]:
for name, s in [("Winning teams", win_sample), ("Losing teams", lose_sample)]:
    m, sem = s.mean(), st.sem(s)
    ci = st.t.interval(0.95, df=len(s) - 1, loc=m, scale=sem)
    print(f"{name}: mean = {m:.3f}, 95% CI = ({ci[0]:.3f}, {ci[1]:.3f})")

Winning teams: mean = 13.933, 95% CI = (11.853, 16.013)
Losing teams: mean = 10.100, 95% CI = (7.978, 12.222)


### Inferential statistics — Two-sample t-Test (Welch's, one-tailed)
H0: μ(win) = μ(lose)  vs.  H1: μ(win) > μ(lose)

In [13]:
t_stat, p_val = st.ttest_ind(win_sample, lose_sample, equal_var=False, alternative='greater')
print(f"t-statistic = {t_stat:.3f}, p-value = {p_val:.4f}")

alpha = 0.05
conclusion = "Reject H0" if p_val < alpha else "Fail to reject H0"
print("Conclusion:", conclusion, "at the 5% significance level.")

t-statistic = 2.638, p-value = 0.0053
Conclusion: Reject H0 at the 5% significance level.


In [14]:
# diff = win_sample.mean() - lose_sample.mean()
# sig = "statistically significant" if p_val < alpha else "not statistically significant"
# # print(
# #     f"Interpretation: In the sample, winning teams averaged {win_sample.mean():.2f} shots "
# #     f"vs {lose_sample.mean():.2f} for losing teams (difference = {diff:.2f}). "
# #     f"With p = {p_val:.4f}, this result is {sig} at the 5% level, so we {conclusion.lower()} "
# #     f"that winning teams take significantly more shots than losing teams."
# # )

**Interpretation**: Winning team average have more shots than losing team